In [ ]:
import os

## Use the code below to run manually ##
# input_folder = 'Outputs/Final_Output_per_scenario'
# output_path = 'Outputs/Summaries'
########################################

## Use the code below to run with the snakemake workflow ##
input_folder = snakemake.params.input_folder
output_path = snakemake.params.output_path
###########################################################
os.makedirs(output_path, exist_ok=True)

## Giving mineral and stage summary per country

In [ ]:
import pandas as pd
import os

def process_all_csv_files_in_folder(input_folder, output_path):
    # List all .csv files in the folder
    csv_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]

    if not csv_files:
        print("No .csv files found in the specified folder.")
        return

    # Process each CSV file
    for csv_file in csv_files:
        try:
            print(f"Processing file: {csv_file}")
            process_data(input_folder, csv_file, output_path)
        except Exception as e:
            print(f"Error processing file {csv_file}: {e}")

def process_data(input_folder, input_file, output_path):
    # Full path to the input file
    input_path = os.path.join(input_folder, input_file)
    scenario = input_file.split(".")[0]

    # Read the original CSV file
    data = pd.read_csv(input_path)

    # Identify the relevant columns
    result_columns = ["Final_Investment_USD", "Req_Capacity_kW", "Est_dem_kWh_year", "Emissions_tCO2eq"]
    demand_columns = [col for col in data.columns if col.startswith("demandper_")]
    data["ISO3"] = data["iso3"]  # Assuming ISO3 is already present

    # Calculate the annual operational expenditure (annual_opex)
    data["annual_opex"] = data["total_demand_postan"] * data["Est_lcoe_$perkWh"]

    # Extend the result columns to include annual_opex
    result_columns.append("annual_opex")

    # Ensure numeric values in relevant columns
    for col in result_columns + demand_columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')

    # Disaggregate results by mineral and stage
    disaggregated_results = pd.DataFrame()
    for result_col in result_columns:
        for demand_col in demand_columns:
            mineral_stage = demand_col.replace("demandper_", "")  # Extract mineral and stage
            disaggregated_results[f"{result_col}_{mineral_stage}"] = (
                data[demand_col] * data[result_col]
            )

    #print (disaggregated_results)
    
    # Append the disaggregated results to the original data
    full_data = pd.concat([data, disaggregated_results], axis=1)

    # Save the full data with disaggregated results to a CSV file
    disaggregated_csv_output = os.path.join(input_folder, f"{scenario}_processed.csv")
    full_data.to_csv(disaggregated_csv_output, index=False)

    #print (disaggregated_results)
    
    # Prepare the summary table: Group by mineral, stage, and ISO3
    grouped_data = disaggregated_results.copy()
    grouped_data["ISO3"] = data["ISO3"]  # Include ISO3 for grouping
    grouped_data.columns = grouped_data.columns.str.replace("Final_Investment_USD_", "investment_", regex=False)
    grouped_data.columns = grouped_data.columns.str.replace("Req_Capacity_kW_", "capacity_", regex=False)
    grouped_data.columns = grouped_data.columns.str.replace("Est_dem_kWh_year_", "demand_", regex=False)
    grouped_data.columns = grouped_data.columns.str.replace("Emissions_tCO2eq_", "emissions_", regex=False)
    grouped_data.columns = grouped_data.columns.str.replace("annual_opex_", "opex_", regex=False)

    #print (grouped_data)
    
    # Melt the DataFrame to a long format for grouping
    long_format = grouped_data.melt(
        id_vars=["ISO3"],
        var_name="metric_stage",
        value_name="value"
    )

    # Extract "metric", "mineral", and "stage" from "metric_stage"
    long_format[["metric", "mineral", "stage"]] = long_format["metric_stage"].str.extract(r"(.+)_(.+)_(.+)")

    # Group by mineral, stage, and ISO3, and sum the values
    summary_table_stage = long_format.groupby(["mineral", "stage", "ISO3", "metric"]).sum().unstack(level="metric")


    # Flatten the multi-index columns
    summary_table_stage.columns = ["_".join(col).strip() for col in summary_table_stage.columns.values]
    summary_table_stage.reset_index(inplace=True)
    
    # Rename columns to match original names
    column_mapping = {
        "value_capacity": "Req_Capacity_kW",
        "value_demand": "Est_dem_kWh_year",
        "value_emissions": "Emissions_tCO2eq",
        "value_investment": "Final_Investment_USD",
        "value_opex": "annual_opex"
    }
    summary_table_stage.rename(columns=column_mapping, inplace=True)
    
    #print (summary_table_stage)

    # Additional summary table excluding stage
    summary_table_no_stage = long_format.groupby(["mineral", "ISO3", "metric"]).sum().unstack(level="metric")
    summary_table_no_stage.rename(columns=column_mapping, inplace=True)
    summary_table_no_stage.columns = ["_".join(col).strip() for col in summary_table_no_stage.columns.values]
    summary_table_no_stage.reset_index(inplace=True)
    
    summary_table_no_stage.rename(columns=column_mapping, inplace=True)

    # Export both summary tables to an Excel file
    summary_xlsx_output = os.path.join(output_path, f"{os.path.splitext(input_file)[0]}_mineral_summary.xlsx")
    with pd.ExcelWriter(summary_xlsx_output) as writer:
        summary_table_stage.to_excel(writer, sheet_name="By Mineral and Stage", index=False)
        summary_table_no_stage.to_excel(writer, sheet_name="By Mineral Only", index=False)

    #print(f"Processed file saved to: {disaggregated_csv_output}")
    #print(f"Summary table saved to: {summary_xlsx_output}")

## To run all files 
process_all_csv_files_in_folder(input_folder, output_path)

### To run only for a file
#process_data(
#    input_folder=r"C:\Users\alexl\Dropbox\Self-employment\Imperial work\Mineral Work\gep-onsset-gep-climate-2023\output\Final_Output_per_scenario",  # Replace with the actual folder path
#    input_file="2022_baseline_country_unconstrained.csv")  # Replace with the actual file name


## Giving country grid Vs OGS summary

In [ ]:
import pandas as pd
import glob
import os

# Define the folder containing the files
folder_path = input_folder  # Replace with your desired folder path if needed

# Find all CSV files with the "_processed" suffix
csv_files = glob.glob(os.path.join(folder_path, '*_processed.csv'))

# Process each file
for file_path in csv_files:
    # Extract the base file name for output
    base_name = os.path.basename(file_path).replace('_processed.csv', '')
    
    # Load the CSV file
    data = pd.read_csv(file_path)
    
    # Check for necessary columns
    if "ISO3" in data.columns and "Decision" in data.columns:
        # Perform the analysis
        country_decision_summary = (
            data.groupby(["ISO3", "Decision"])
            .agg(
                Final_Investment_USD=("Final_Investment_USD", "sum"),
                Req_Capacity_kW=("Req_Capacity_kW", "sum"),
                avg_lcoe_perkWh=("Est_lcoe_$perkWh", "mean"),
                Emissions_tCO2eq=("Emissions_tCO2eq", "sum"),
                annual_opex_USD=("annual_opex", "sum"),
                annual_demand_kWh=("total_demand_postan", "sum")
            )
            .reset_index()
        )
        
        # Rename columns to match the specified names
        country_decision_summary.rename(columns={
            "Decision": "Decision",
            "Final_Investment_USD": "Final_Investment_USD",
            "Req_Capacity_kW": "Req_Capacity_kW",
            "avg_lcoe_$perkWh": "avg_lcoe_$perkWh",
            "Emissions_tCO2eq": "Emissions_tCO2eq",
            "annual_opex_USD": "annual_opex_USD",
            "annual_demand_kWh": "annual_demand_kWh"
        }, inplace=True)
        
        # Define output file path
        output_file_path = os.path.join(output_path, f'{base_name}_tech_summary.xlsx')
        
        # Save the summary to an Excel file
        country_decision_summary.to_excel(output_file_path, index=False)
        
        #print(f"Analysis for {file_path} saved to {output_file_path}")
    else:
        print(f"Skipping {file_path}: Required columns 'ISO3' and 'Decision' not found.")


## Overall summary comparison (high level)

In [ ]:
import pandas as pd
import glob
import os

# Define the folder containing the summary files
folder_path = output_path  # Replace with the folder containing the summary Excel files

# Find all Excel files with the "_summary.xlsx" suffix
summary_files = glob.glob(os.path.join(folder_path, '*_tech_summary.xlsx'))

# Initialize a list to store data from all files
all_scenarios = []

# Process each summary file
for file_path in summary_files:
    # Extract scenario name from the file name
    scenario_name = os.path.basename(file_path).replace('_tech_summary.xlsx', '')
    
    # Load the summary Excel file
    summary_data = pd.read_excel(file_path)
    
    # Add a column for the scenario name
    summary_data['Scenario'] = scenario_name
    
    # Append the data to the list
    all_scenarios.append(summary_data)

# Combine all scenarios into a single DataFrame
combined_data = pd.concat(all_scenarios, ignore_index=True)

# Group by Scenario and calculate total metrics for comparison
scenario_comparison = combined_data.groupby("Scenario").agg(
    Total_Investment_USD=("Final_Investment_USD", "sum"),
    Total_Capacity_kW=("Req_Capacity_kW", "sum"),
    Total_Emissions_tCO2eq=("Emissions_tCO2eq", "sum"),
    Total_Annual_Opex_USD=("annual_opex_USD", "sum"),
    Total_Annual_Demand_kWh=("annual_demand_kWh", "sum")
).reset_index()

# Save the comparison to an Excel file
comparison_output_path = os.path.join(output_path, 'tech_scenario_comparison.xlsx')
scenario_comparison.to_excel(comparison_output_path, index=False)

# Print confirmation
print(f"Scenario comparison saved to {comparison_output_path}")

# Optional: Display the comparison for review
#print(scenario_comparison)


In [ ]:
import pandas as pd
import glob
import os

# Define the folder containing the summary files
folder_path = output_path # Replace with the folder containing the summary Excel files

# Find all Excel files with the "_summary.xlsx" suffix
summary_files = glob.glob(os.path.join(folder_path, '*_tech_summary.xlsx'))

# Initialize a list to store data from all files
all_scenarios = []

# Process each summary file
for file_path in summary_files:
    # Extract scenario name from the file name
    scenario_name = os.path.basename(file_path).replace('_tech_summary.xlsx', '')
    
    # Load the summary Excel file
    summary_data = pd.read_excel(file_path)
    
    # Add a column for the scenario name
    summary_data['Scenario'] = scenario_name
    
    # Append the data to the list
    all_scenarios.append(summary_data)

# Combine all scenarios into a single DataFrame
combined_data = pd.concat(all_scenarios, ignore_index=True)

# 1. Summary without splitting by Decision
overall_scenario_comparison = combined_data.groupby("Scenario").agg(
    Total_Investment_USD=("Final_Investment_USD", "sum"),
    Total_Capacity_kW=("Req_Capacity_kW", "sum"),
    Total_Emissions_tCO2eq=("Emissions_tCO2eq", "sum"),
    Total_Annual_Opex_USD=("annual_opex_USD", "sum"),
    Total_Annual_Demand_kWh=("annual_demand_kWh", "sum")
).reset_index()

# 2. Summary with splitting by Decision
scenario_decision_comparison = combined_data.groupby(["Scenario", "Decision"]).agg(
    Total_Investment_USD=("Final_Investment_USD", "sum"),
    Total_Capacity_kW=("Req_Capacity_kW", "sum"),
    Total_Emissions_tCO2eq=("Emissions_tCO2eq", "sum"),
    Total_Annual_Opex_USD=("annual_opex_USD", "sum"),
    Total_Annual_Demand_kWh=("annual_demand_kWh", "sum")
).reset_index()

# Save both summaries to the same Excel file with different sheets
output_file_path = os.path.join(output_path, 'all_scenarios_comparison_combined.xlsx')
with pd.ExcelWriter(output_file_path) as writer:
    overall_scenario_comparison.to_excel(writer, sheet_name='Overall Summary', index=False)
    scenario_decision_comparison.to_excel(writer, sheet_name='Decision Split Summary', index=False)

# Print confirmation
print(f"Combined scenario comparison saved to {output_file_path}")
